# Superstore Retail Sales Analysis

This notebook covers the initial data loading, data quality assessment, cleaning, and exploratory analysis of the Superstore retail dataset.

The prepared dataset will be used for further **SQL analysis** and **Power BI reporting**.


In [2]:
import pandas as pd

df = pd.read_csv(r"C:\projects\superstore\train.csv")

## 1. Initial Data Inspection

First, we inspect the dataset structure, sample records, data types, and dimensions to understand the data before performing any transformations.


In [3]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

## 2. Data Quality Check

Before cleaning the dataset, we check for:

* Missing values
* Duplicate records
* Potential data quality issues

This helps ensure that the subsequent analysis is based on a consistent dataset.


In [5]:
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

## 3. Data Cleaning

### Date Formatting

The `Order Date` and `Ship Date` columns are converted to the `datetime` data type to enable time-based analysis and feature extraction.


In [7]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

df[['Order Date', 'Ship Date']].dtypes

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object

### Handling Missing Values

The `Postal Code` column contains missing values. Since this field is primarily required for geographic analysis, we first evaluate the proportion of missing records before deciding how to handle them.


In [8]:
missing_postal_share = df['Postal Code'].isnull().mean() * 100

print(f"Missing Postal Code values: {missing_postal_share:.2f}%")
                                            

Missing Postal Code values: 0.11%


The missing values in `Postal Code` account for only a very small share of the dataset. Since these records cannot provide complete geographic information and represent a negligible portion of the data, they are removed from the dataset.

This prevents incomplete geographic records from affecting the regional analysis.


In [9]:
df = df.dropna(subset=['Postal Code']).copy()

print(f"Remaining rows: {len(df):,}")

Remaining rows: 9,789


After removing the incomplete records, we perform a final check to confirm that the dataset no longer contains missing values in `Postal Code` and that no duplicate records were introduced or retained.


In [10]:
print(f"Missing Postal Code values: {df['Postal Code'].isnull().sum():,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

Missing Postal Code values: 0
Duplicate rows: 0


## 4. Key Sales Metrics

We calculate the main sales KPIs to establish a high-level overview of the business:

* Total revenue
* Number of unique orders
* Average Order Value (AOV)

AOV is calculated at the **order level** rather than as the average of individual sales lines, since a single order may contain multiple products.


In [11]:
order_sales = df.groupby('Order ID')['Sales'].sum()

total_revenue = order_sales.sum()
number_of_orders = order_sales.size
average_order_value = order_sales.mean()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Number of Orders: {number_of_orders:,}")
print(f"Average Order Value: ${average_order_value:,.2f}")

Total Revenue: $2,252,607.41
Number of Orders: 4,916
Average Order Value: $458.22


## 5. Sales by Product Category

Next, we compare total revenue across product categories to identify the strongest contributors to overall sales.


In [12]:
category_sales = (
    df.groupby('Category', as_index=False)['Sales']
      .sum()
      .sort_values('Sales', ascending=False)
)

category_sales

,Category,Sales
2,Technology,825856.1130
0,Furniture,723538.4757
1,Office Supplies,703212.8240


## 6. Feature Engineering

To support time-based analysis, we extract the year and month from `Order Date`.

These features will be used later to analyze sales trends and seasonality.


In [13]:
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month

### Sales by Year and Month

We calculate total sales by year and month to prepare the dataset for further time-series analysis.


In [14]:
yearly_sales = (
    df.groupby('Year', as_index=False)['Sales']
      .sum()
      .sort_values('Year')
)

monthly_sales = (
    df.groupby('Month', as_index=False)['Sales']
      .sum()
      .sort_values('Month')
)

yearly_sales

,Year,Sales
0,2015,479856.2081
1,2016,454315.9054
2,2017,597225.4900
3,2018,721209.8092


In [15]:
monthly_sales

,Month,Sales
0,1,91982.1396
1,2,59371.1154
2,3,197573.5872
3,4,134988.2506
4,5,154086.7237
5,6,145837.5233
6,7,145535.6890
7,8,157315.9270
8,9,300103.4117
9,10,199496.2947


## 7. Export Cleaned Dataset

The cleaned and prepared dataset is exported to a CSV file for further **SQL analysis** and **Power BI reporting**.


In [16]:
df.to_csv(r"C:\Users\whity\OneDrive\Рабочий стол\GitHub\Superstore_Sales_Analysis\superstore_cleaned.csv", index=False)